# 06 — Team Z-Scores & Qualities

Compute within-league z-scores for all team stats, plus **projected z-scores** (same raw stats vs next season's league population).  
Then compute 7 team style qualities using Twelve's formulas.

**Scope:**
- 8 leagues: PL, La Liga, Serie A, Ligue 1, Eredivisie, Bundesliga, Primeira Liga, Superliga Turquía
- 6 seasons: 2019/20 through 2024/25 (integer: 2019–2024)
- Also load 2019/20 (season=2019) for projecting into 2020/21
- 2024/25: current z-scores only (no projection forward)

In [ ]:
import pandas as pd
import numpy as np
import warnings
warnings.filterwarnings('ignore')

DATA_ROOT = '/Users/jorgepadilla/Documents/Documents - Jorge\u2019s MacBook Air/thesis_data'
ts = pd.read_parquet(f'{DATA_ROOT}/raw_data/Teams_stats/team_stats_season.parquet')
print(f'Loaded team_stats_season: {ts.shape}')

## 1. Filter to Target Leagues & Seasons

In [ ]:
LEAGUES = {
    364: 'Premier League',
    795: 'La Liga',
    524: 'Serie A',
    412: 'Ligue 1',
    635: 'Eredivisie',
    426: 'Bundesliga',
    707: 'Primeira Liga',
    852: 'Super Lig',
}

# Seasons 2019-2024 (= 2019/20 through 2024/25)
# 2019 needed for projection into 2020
SEASONS = [2019, 2020, 2021, 2022, 2023, 2024]

df = ts[(ts['competition_id'].isin(LEAGUES.keys())) & (ts['season'].isin(SEASONS))].copy()
df['league_name'] = df['competition_id'].map(LEAGUES)

print(f'Filtered: {len(df):,} rows')
print(f'\nTeams per (league, season):')
pivot = df.groupby(['league_name', 'season'])['team_id'].nunique().unstack(fill_value=0)
print(pivot.to_string())

## 2. Define Quality Formulas & Metric Mappings

From `twelve_qualities/team_qualities.txt`

In [ ]:
# Mapping: Twelve formula variable -> team_stats_season column name
METRIC_MAP = {
    # DEFENCE
    'TEAM_DEFENSIVE_INTENSITY': 'defensive_intensity',
    'PPDA': 'ppda',
    'FINAL_THIRD_RECOVERIES_PCT': 'final_third_recoveries_pct',
    'TEAM_DEFENSIVE_LINE_HEIGHT': 'defensive_action_height_m',
    # DEFENSIVE_TRANSITION
    'RECOVERIES_WITHIN_5S_PCT': 'recoveries_within_5s_pct',
    'TIME_TO_DEFENSIVE_ACTION_AFTER_LOSS_IN_AH': 'time_to_defensive_action_after_loss_att_half_s',
    'TIME_TO_DEFENSIVE_ACTION_AFTER_LOSS_IN_OH': 'time_to_defensive_action_after_loss_own_half_s',
    # ATTACKING_TRANSITION
    'POSSESSIONS_RETAINED_AFTER_5S_PCT': 'possessions_retained_after_5s_pct',
    'FINAL_THIRD_ENTRIES_WITHIN_10S_AFTER_OH_RECOVERY_PCT': 'final_third_entry_within_10s_after_recovery_own_half_pct',
    'FIRST_PASS_FORWARD_PCT': 'first_pass_forward_after_recovery_own_half_pct',
    'TIME_TO_FORWARD_PASS_AFTER_OH_RECOVERY': 'median_time_to_first_forward_pass_own_half_s',
    # ATTACK
    'LONG_BALL_PCT': 'long_ball_pct',
    'FORWARD_PASSES_FROM_MIDDLE_THIRD_PCT': 'forward_passes_from_middle_third_pct',
    'BUILDUP_FROM_GOALKICK_PCT': 'buildups_from_goalkicks_pct',
    # PENETRATION
    'BOX_ENTRIES_FROM_CARRIES_PCT': 'box_entries_from_carries_pct',
    'BOX_ENTRIES_FROM_CROSSES_PCT': 'box_entries_from_crosses_pct',
    'CROSSES_PER_FINAL_THIRD_ENTRY': 'crosses_per_final_third_possession',
    # CHANCE_CREATION
    'SHOTS_PER_FINAL_THIRD_PASS': 'shots_per_final_third_pass',
    'SHOTS_FROM_DIRECT_ATTACKS_PCT': 'shots_from_direct_attacks_pct',
    'SHOTS_FROM_SUSTAINED_ATTACKS_PCT': 'shots_from_sustained_attacks_pct',
    # OUTCOME
    'XPOINTS': 'xpts',
    'POINTS': 'points',
}

# Quality formulas: quality_name -> {column_name: weight}
QUALITIES = {
    'DEFENCE': {
        'defensive_intensity': 1.0,
        'ppda': 1.0,
        'final_third_recoveries_pct': 1.0,
        'defensive_action_height_m': 1.0,
    },
    'DEFENSIVE_TRANSITION': {
        'recoveries_within_5s_pct': 1.0,
        'time_to_defensive_action_after_loss_att_half_s': 2.0,
        'time_to_defensive_action_after_loss_own_half_s': 1.0,
    },
    'ATTACKING_TRANSITION': {
        'possessions_retained_after_5s_pct': 0.5,
        'final_third_entry_within_10s_after_recovery_own_half_pct': 0.5,
        'first_pass_forward_after_recovery_own_half_pct': 1.0,
        'median_time_to_first_forward_pass_own_half_s': 0.5,
    },
    'ATTACK': {
        'long_ball_pct': 2.0,
        'forward_passes_from_middle_third_pct': 1.0,
        'buildups_from_goalkicks_pct': 1.0,
    },
    'PENETRATION': {
        'box_entries_from_carries_pct': 2.0,
        'box_entries_from_crosses_pct': 2.0,
        'crosses_per_final_third_possession': 1.0,
    },
    'CHANCE_CREATION': {
        'shots_per_final_third_pass': 1.0,
        'shots_from_direct_attacks_pct': 2.0,
        'shots_from_sustained_attacks_pct': 2.0,
    },
    'OUTCOME': {
        'xpts': 1.5,
        'points': 1.0,
    },
}

# Metrics where higher_is_better=False in Twelve's team.py (z-score gets negated)
# Only these 2 quality-formula metrics have that flag; the time metrics only have
# style_of_play_left=True which is a UI flag, not a z-score direction flag.
LOWER_IS_BETTER = {
    'ppda',            # higher_is_better=False — lower PPDA = more pressing
    'long_ball_pct',   # higher_is_better=False — lower long ball % = more build-up
}

# All metric columns used in qualities
quality_metrics = sorted(set(METRIC_MAP.values()))
print(f'Quality metrics: {len(quality_metrics)}')
print(f'Lower-is-better: {len(LOWER_IS_BETTER)}')
print(f'Qualities: {list(QUALITIES.keys())}')

# Verify all metrics exist in the dataframe
missing = [m for m in quality_metrics if m not in df.columns]
if missing:
    print(f'\nWARNING: Missing columns: {missing}')
else:
    print(f'\nAll {len(quality_metrics)} quality metrics found in dataframe')

## 3. Compute Current Z-Scores (within league + season)

For each metric, z-score = (value - league_season_mean) / league_season_std  
For "lower is better" metrics, negate the z-score.

In [ ]:
id_cols = ['team_id', 'competition_id', 'season']
metric_cols = [c for c in ts.columns if c not in id_cols]

# Compute league+season reference stats for ALL metrics
ref_stats = df.groupby(['competition_id', 'season'])[metric_cols].agg(['mean', 'std'])

# Compute current z-scores
z_current = pd.DataFrame(index=df.index)

for col in metric_cols:
    mean_map = ref_stats[(col, 'mean')]
    std_map = ref_stats[(col, 'std')]
    
    # Map mean/std to each row via (competition_id, season)
    keys = list(zip(df['competition_id'], df['season']))
    means = pd.Series([mean_map.get(k, np.nan) for k in keys], index=df.index)
    stds = pd.Series([std_map.get(k, np.nan) for k in keys], index=df.index)
    
    z = (df[col] - means) / stds.replace(0, np.nan)
    
    # Negate for lower-is-better metrics
    if col in LOWER_IS_BETTER:
        z = -z
    
    z_current[f'z_{col}'] = z

print(f'Current z-scores computed: {z_current.shape}')
print(f'\nSample z-score distributions (should be ~mean=0, std=1):')
sample_z = ['z_goals', 'z_xg', 'z_ppda', 'z_defensive_intensity', 'z_points']
for zc in sample_z:
    if zc in z_current.columns:
        vals = z_current[zc].dropna()
        print(f'  {zc:40s} mean={vals.mean():+.4f}  std={vals.std():.4f}  n={len(vals)}')

## 4. Compute Projected Z-Scores (raw stats vs NEXT season's league population)

For a team in season T: take its raw stats and z-score them against the distribution of its league in season T+1.  
This answers: "if this team played exactly the same way next season, how would it rank?"

- Seasons 2019-2023: projected to 2020-2024
- Season 2024: **no projection** (no season 2025 data for these leagues)

In [ ]:
z_projected = pd.DataFrame(index=df.index)

for col in metric_cols:
    mean_map = ref_stats[(col, 'mean')]
    std_map = ref_stats[(col, 'std')]
    
    projected_z = pd.Series(np.nan, index=df.index)
    
    for idx, row in df.iterrows():
        next_season = row['season'] + 1
        comp = row['competition_id']
        key = (comp, next_season)
        
        # Only project if next season exists in our data
        if key in mean_map.index:
            m = mean_map[key]
            s = std_map[key]
            if s > 0 and pd.notna(row[col]):
                z = (row[col] - m) / s
                if col in LOWER_IS_BETTER:
                    z = -z
                projected_z.at[idx] = z
    
    z_projected[f'z_proj_{col}'] = projected_z

# Check how many rows got projections
has_proj = z_projected.notna().any(axis=1).sum()
no_proj = z_projected.isna().all(axis=1).sum()
print(f'Projected z-scores computed: {z_projected.shape}')
print(f'Rows WITH projection: {has_proj:,}')
print(f'Rows WITHOUT projection (season 2024): {no_proj:,}')

# Verify: should be exactly the season 2024 rows
n_2024 = (df['season'] == 2024).sum()
print(f'Season 2024 rows: {n_2024} (should match rows without projection)')

## 5. Compute Team Style Qualities

Each quality = weighted sum of z-scores for its component metrics.

In [ ]:
def compute_qualities(z_df, prefix, z_col_prefix):
    """Compute 7 team qualities from a z-score dataframe.
    
    Args:
        z_df: DataFrame with z-score columns
        prefix: output column prefix (e.g., 'q_' or 'q_proj_')
        z_col_prefix: prefix of z-score columns (e.g., 'z_' or 'z_proj_')
    """
    result = pd.DataFrame(index=z_df.index)
    
    for quality_name, weights in QUALITIES.items():
        q = pd.Series(0.0, index=z_df.index)
        total_weight = 0
        any_valid = pd.Series(False, index=z_df.index)
        
        for metric_col, weight in weights.items():
            z_col = f'{z_col_prefix}{metric_col}'
            if z_col in z_df.columns:
                valid_mask = z_df[z_col].notna()
                any_valid = any_valid | valid_mask
                q += weight * z_df[z_col].fillna(0)
                total_weight += weight
        
        # Normalize by total weight so qualities are comparable
        if total_weight > 0:
            q = q / total_weight
        
        # If ALL z-scores were NaN for a row, quality should be NaN too
        q[~any_valid] = np.nan
        
        result[f'{prefix}{quality_name}'] = q
    
    return result

# Current qualities
q_current = compute_qualities(z_current, prefix='q_', z_col_prefix='z_')
print('CURRENT QUALITIES')
print(q_current.describe().round(3).to_string())

# Projected qualities
q_projected = compute_qualities(z_projected, prefix='q_proj_', z_col_prefix='z_proj_')
print(f'\nPROJECTED QUALITIES')
print(q_projected.describe().round(3).to_string())

## 6. Assemble Output Parquet

In [ ]:
# Assemble: IDs + league name + raw stats + current z-scores + projected z-scores + qualities
output = pd.concat([
    df[['team_id', 'competition_id', 'season', 'league_name']].reset_index(drop=True),
    df[metric_cols].reset_index(drop=True),
    z_current.reset_index(drop=True),
    z_projected.reset_index(drop=True),
    q_current.reset_index(drop=True),
    q_projected.reset_index(drop=True),
], axis=1)

print(f'Output shape: {output.shape}')
print(f'\nColumn groups:')
n_raw = len(metric_cols)
n_z = len([c for c in output.columns if c.startswith('z_') and not c.startswith('z_proj_')])
n_zp = len([c for c in output.columns if c.startswith('z_proj_')])
n_q = len([c for c in output.columns if c.startswith('q_') and not c.startswith('q_proj_')])
n_qp = len([c for c in output.columns if c.startswith('q_proj_')])
print(f'  IDs + league_name:     4')
print(f'  Raw metrics:           {n_raw}')
print(f'  Current z-scores:      {n_z}')
print(f'  Projected z-scores:    {n_zp}')
print(f'  Current qualities:     {n_q}')
print(f'  Projected qualities:   {n_qp}')
print(f'  TOTAL:                 {output.shape[1]}')

## 7. Validation

Sanity checks to make sure the z-scores and qualities make sense.

In [ ]:
import matplotlib.pyplot as plt

# Check 1: Current z-scores should have mean~0, std~1 within each (league, season)
print('CHECK 1: Current z-score means by (league, season) — should be ~0')
z_means = output.groupby(['league_name', 'season'])['z_goals'].mean()
print(f'  z_goals mean range: [{z_means.min():.6f}, {z_means.max():.6f}]')

z_stds = output.groupby(['league_name', 'season'])['z_goals'].std()
print(f'  z_goals std range:  [{z_stds.min():.4f}, {z_stds.max():.4f}]')

# Check 2: Projected z-scores should NOT have mean~0 within current season
# (because they're scored against a DIFFERENT population)
proj_means = output[output['season'] < 2024].groupby(['league_name', 'season'])['z_proj_goals'].mean()
print(f'\nCHECK 2: Projected z-score means (should NOT be exactly 0):')
print(f'  z_proj_goals mean range: [{proj_means.min():.4f}, {proj_means.max():.4f}]')

# Check 3: Season 2024 should have NaN projected z-scores
s2024 = output[output['season'] == 2024]
proj_cols = [c for c in output.columns if c.startswith('z_proj_')]
pct_nan = s2024[proj_cols].isna().all(axis=1).mean()
print(f'\nCHECK 3: Season 2024 projected z-scores all NaN: {pct_nan:.1%}')

In [ ]:
# Check 4: Spot-check a known team
# Pick a Premier League team across seasons
pl = output[output['competition_id'] == 364].copy()
# Find a team that appears in all 6 seasons
team_counts = pl.groupby('team_id')['season'].nunique()
stable_team = team_counts[team_counts == 6].index[0]

team_data = pl[pl['team_id'] == stable_team].sort_values('season')
team_name = f'Team {stable_team}'

print(f'SPOT CHECK: {team_name} (Premier League, all 6 seasons)')
print(f'\nRaw goals per season:')
for _, row in team_data.iterrows():
    proj_str = f'  z_proj={row["z_proj_goals"]:+.3f}' if pd.notna(row.get('z_proj_goals')) else '  z_proj=N/A'
    print(f'  {int(row["season"])}/{int(row["season"])+1}: goals={row["goals"]:.2f}  z_current={row["z_goals"]:+.3f}{proj_str}')

print(f'\nCurrent qualities:')
q_cols = [c for c in output.columns if c.startswith('q_') and not c.startswith('q_proj_')]
print(team_data[['season'] + q_cols].to_string(index=False))

print(f'\nProjected qualities:')
qp_cols = [c for c in output.columns if c.startswith('q_proj_')]
print(team_data[['season'] + qp_cols].to_string(index=False))

In [ ]:
# Visualize: current vs projected quality for one quality across PL teams
fig, axes = plt.subplots(2, 4, figsize=(18, 8))
axes = axes.flat

for i, q_name in enumerate(QUALITIES.keys()):
    ax = axes[i]
    # Filter to seasons with both current and projected
    valid = output[(output['season'] < 2024) & (output['competition_id'] == 364)]
    ax.scatter(valid[f'q_{q_name}'], valid[f'q_proj_{q_name}'],
              alpha=0.4, s=15, color='steelblue')
    ax.plot([-3, 3], [-3, 3], 'r--', lw=1, alpha=0.5)
    ax.set_xlabel(f'Current', fontsize=8)
    ax.set_ylabel(f'Projected', fontsize=8)
    ax.set_title(q_name, fontsize=10)
    ax.tick_params(labelsize=7)
    r = valid[[f'q_{q_name}', f'q_proj_{q_name}']].corr().iloc[0, 1]
    ax.text(0.05, 0.95, f'r={r:.3f}', transform=ax.transAxes, fontsize=8, va='top')

# Hide extra subplot
axes[7].set_visible(False)

plt.suptitle('Premier League: Current vs Projected Team Qualities', fontsize=12)
plt.tight_layout()
plt.show()

## 8. Save Output

In [ ]:
out_path = f'{DATA_ROOT}/processed_data/model_dataset/v_1/team_stats_z_scores_qualities.parquet'
output.to_parquet(out_path, index=False)
print(f'Saved: {out_path}')
print(f'Shape: {output.shape}')
print(f'Size: {output.memory_usage(deep=True).sum() / 1e6:.1f} MB')

# Final summary
print(f'\n=== SUMMARY ===')
print(f'Leagues: {len(LEAGUES)}')
print(f'Seasons: {sorted(output["season"].unique())}')
print(f'Total rows: {len(output):,}')
print(f'Columns: {output.shape[1]}')
print(f'  - Raw metrics: {n_raw}')
print(f'  - Current z-scores: {n_z}')
print(f'  - Projected z-scores: {n_zp}')
print(f'  - Current qualities: {n_q}')
print(f'  - Projected qualities: {n_qp}')